# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² Croissant dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset provides protein-level mass spectrometry data for extracellular vesicles isolated from human mast cells, and is described in a [Croissant JSON-LD schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure required library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the Croissant dataset metadata and initialize the dataset with `mlcroissant`. This allows exploring all available record sets and data fields as defined by their `@id` values.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Display name and description
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview

Let's review the available record sets, their fields, and columns along with their `@id` values.

The dataset uses record set and field `@id`s to refer to each data entity uniquely. We will gather all available record sets and for each list their available fields and columns (with their `@id`s and semantic names).


In [ ]:
# --- List all record sets and their fields/columns by @id ---

# RecordSet objects are available from metadata.record_sets
record_sets = getattr(metadata, 'record_sets', [])
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"\nRecord Set: {getattr(rs, 'name', '')} (record_set @id={getattr(rs, '@id', '')})")
    fields = getattr(rs, 'fields', [])
    if fields:
        for f in fields:
            print(f"  Field: {getattr(f, 'name', '')} (@id={getattr(f, '@id', '')}, type={getattr(f, 'data_type', '')})")
            columns = getattr(f, 'columns', [])
            for c in columns or []:
                print(f"     Column: {getattr(c, 'name', '')} (@id={getattr(c, '@id', '')})")


## 3. Data Extraction

Let's load all record sets as individual pandas DataFrames for flexible exploratory analysis. We will reference all entities strictly by their `@id` according to Croissant best practice.

> **Tip:** To keep this notebook self-contained, the table below loads all main record sets detected previously.


In [ ]:
# List of detected record set @id values
# You can add or remove @id strings (copy these from the previous cell's printout)
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in getattr(metadata, 'record_sets', [])]
record_set_ids = [rsid for rsid in record_set_ids if rsid]

print(f"Extracting from record sets: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id} columns: {df.columns.tolist()}")
        print(df.head(), '\n')
    except Exception as e:
        print(f"Failed to extract record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

We demonstrate EDA using the primary protein abundance record set. All analysis steps reference fields and columns via their `@id` values, per the Croissant schema.

Typical operations include filtering proteins by abundance, normalizing quantitative fields, and grouping by key attributes.


In [ ]:
# --- EDA: Filtering and Normalization by @id ---

# Example: Choose the first detected non-empty record_set for demonstration
primary_rs_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        primary_rs_id = rsid
        break
if primary_rs_id is None:
    raise RuntimeError('No non-empty record set found!')
print(f'Using record set @id: {primary_rs_id}')
df = dataframes[primary_rs_id]

# Identify a numeric field (@id of a column with float/integer values)
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f'Numeric fields detected: {numeric_candidates}')
if not numeric_candidates:
    raise RuntimeError('No numeric field found in selected record set for demonstration.')
numeric_field_id = numeric_candidates[0]

threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() < df[numeric_field_id].max() else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold} (using @id references):")
print(filtered_df.head())

# Normalization
field_norm = numeric_field_id + '_normalized'
filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
print(filtered_df[[numeric_field_id, field_norm]].head())

# Attempt grouping by a categorical field (@id)
categorical_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < df.shape[0]/2]
if categorical_candidates:
    group_field_id = categorical_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field_id} (using @id):")
    print(grouped_df.head())
else:
    print('No suitable categorical field found for grouping.')

## 5. Visualization

Visualize the distribution of the selected numeric field for the protein abundance record set. All axes reference field/column `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field (by @id)
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f'Histogram of {numeric_field_id}')
plt.xlabel(f'Field @id: {numeric_field_id}')
plt.ylabel('Count')
plt.show()

# If a grouping field was chosen, show boxplot by group
if 'group_field_id' in locals():
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id} (by @id)')
    plt.xlabel(f'Group field @id: {group_field_id}')
    plt.ylabel(f'Numeric field @id: {numeric_field_id}')
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, explore, and process a Croissant-compliant dataset using the `mlcroissant` Python library.
- All field and record set references use the Croissant entity `@id` values to ensure programmatic robustness and traceability.
- The approach enables flexible exploration (filtering, normalization, visualization) using only schema-level identifiers.

You can further customize code cells to support your own workflows by substituting your target record set and field `@id`s.

---
**For more information on Croissant:**
- [GitHub: mlcroissant](https://github.com/mlcommons/croissant)
- [FAIR² Dataset DOI: 10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
